# Van der Pol equation

In [ ]:
import numpy as np
from scipy.optimize import root
from scipy.integrate import solve_ivp
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.templates.default = "seaborn"

We consider the following problem :

$$
\left\{ 
\begin{aligned} 
{\mathrm d}_t x & = \epsilon^2 \left(y-\frac{x^3}{3}+x \right)\\ 
{\mathrm d}_t y & = -x 
\end{aligned} 
\right. 
\qquad{} \text{with } \epsilon > 0
$$

In [ ]:
class vanderpol_model:

    def __init__(self, eps):
        self.eps = eps

    def fcn(self, t, y):
        y1, y2 = y
        eps = self.eps
        y1_dot = (eps**2)*(y2-y1**3/3+y1)
        y2_dot = -y1
        return np.array([y1_dot, y2_dot])

    def jac(self, t, y):
        y1, y2 = y
        eps = self.eps
        return np.array([[eps**2*(-y1**2 + 1), eps**2], [-1, 0]])

## Quasi-exact solution

The quasi-exact solution is obtained by using an explicit Runge-Kutta method of order 8 with stepsize control and fine tolerances due to Dormand and Prince.

In [ ]:
tini = 0. 
tend = 10.

yini = (0.5, 0)

vdpm = vanderpol_model(eps=1)
fcn = vdpm.fcn  
sol = solve_ivp(fcn, (tini, tend), yini, method="DOP853", rtol=1e-09, atol=1e-12)

fig = make_subplots(rows=2, cols=2, specs=[[{"colspan": 2}, None],[{}, None]], 
                    subplot_titles=("Solution","Phase plane"), vertical_spacing=0.2)

eps = np.arange(1,21)
for eps_i in eps:
    vdpm = vanderpol_model(eps=eps_i)
    fcn = vdpm.fcn  
    sol = solve_ivp(fcn, (tini, tend), yini, method="DOP853", rtol=1e-9, atol=1e-12)
    fig.add_trace(go.Scatter(visible=False, x=sol.t, y=sol.y[0], name='x', legendgroup='sol'), row=1, col=1)
    fig.add_trace(go.Scatter(visible=False, x=sol.y[0], y=sol.y[1], showlegend=False, legendgroup='pp',), row=2, col=1)

#slow_manifold = x**3/3 - x
pts = np.linspace(-2.4,2.4,1000)
slow_manifold = pts**3/3 - pts
fig.add_trace(go.Scatter(x=pts, y=slow_manifold, name='slow manifold', legendgroup='pp'), row=2, col=1)

fig.data[0].visible = True
fig.data[1].visible = True

steps = []
for i, eps_i in enumerate(eps):
    args = [{"visible": [(el==2*i) or (el==2*i+1) or (el==len(fig.data)-1) for el in range(len(fig.data))]}]
    step = dict(method="update", label = f"{eps_i}", args=args)
    steps.append(step)
sliders = [dict(currentvalue={"prefix": "eps : "}, steps=steps)]

legend = dict(x=0.0, bgcolor='rgba(0,0,0,0)', tracegroupgap=280)
fig.update_layout(legend=legend, height=700, sliders=sliders)
fig.update_xaxes(title_text='t', row=1)
fig.update_xaxes(title_text='x', row=2)
fig.update_yaxes(title_text='y', row=2)

## Characterization of stiffness

In [ ]:
tini = 0. 
tend = 10.

yini = (0.5, 0)

fig = make_subplots(rows=2, cols=1, vertical_spacing=0.2,
                    subplot_titles=("Real part of eigenvalues","Imaginary part of eigenvalues"))

eps = np.arange(1,21)
for eps_i in eps:
    vdpm = vanderpol_model(eps=eps_i)
    fcn = vdpm.fcn
    jac = vdpm.jac
    sol = solve_ivp(fcn, (tini, tend), yini, method="DOP853", rtol=1e-09, atol=1e-12)
    eig_vals = np.zeros((sol.t.size, 2), dtype=np.complex128)
    for it in range(0,sol.t.size):
        eig_vals[it] = np.linalg.eigvals(jac(0., sol.y[:,it]))
    lambda1 = eig_vals[:, 0]
    lambda2 = eig_vals[:, 1]

    color = fig.layout.template.layout.colorway
    fig.add_trace(go.Scatter(visible=False, x=sol.t, y=np.real(lambda1), line_color=color[0], name='\u03BB1', legendgroup='real',), row=1, col=1)
    fig.add_trace(go.Scatter(visible=False, x=sol.t, y=np.real(lambda2), line_color=color[1], name='\u03BB2', legendgroup='real',), row=1, col=1)
    fig.add_trace(go.Scatter(visible=False, x=sol.t, y=np.imag(lambda1), line_color=color[0], name='\u03BB1', legendgroup='imag',), row=2, col=1)
    fig.add_trace(go.Scatter(visible=False, x=sol.t, y=np.imag(lambda2), line_color=color[1], name='\u03BB2', legendgroup='imag',), row=2, col=1)

for i in range(4): fig.data[i].visible = True

steps = []
for i, eps_i in enumerate(eps):
    args = [{"visible": [(el>=4*i) and (el<=4*i+3) for el in range(len(fig.data))]}]
    step = dict(method="update", label = f"{eps_i}", args=args)
    steps.append(step)
sliders = [dict(currentvalue={"prefix": "eps : "}, steps=steps)]

legend = dict(x=0.0, bgcolor='rgba(0,0,0,0)', tracegroupgap=280)
fig.update_layout(legend=legend, height=700, sliders=sliders)
fig.update_xaxes(title_text='t', row=1)
fig.show()

## Dormand and Price method

### Method of order 5

In order to compute the global error, the quasi-exact solution is obtained by using an explicit Runge-Kutta method of order 5 with stepsize control and fine tolerances due to Dormand and Prince.

In [ ]:
tini = 0. 
tend = 10.

yini = (0.5, 0)

tol = 1e-6

fig = make_subplots(rows=3, cols=1, vertical_spacing=0.1,
                    subplot_titles=(f"Solution x", f"Global error (tol={tol})", f"Time steps (tol={tol})"))

eps = np.arange(1,21)
for eps_i in eps:
    vdpm = vanderpol_model(eps=eps_i)
    fcn = vdpm.fcn
    jac = vdpm.jac
    sol = solve_ivp(fcn, (tini, tend), yini, method="RK45", rtol=tol, atol=tol)
    sol_exa = solve_ivp(fcn, (tini, tend), yini, method="DOP853", rtol=1e-09, atol=1e-12, t_eval=sol.t)
    err = np.abs(sol_exa.y[0]-sol.y[0])
    dt = sol.t[1:]-sol.t[:-1]
    
    fig.add_trace(go.Scatter(visible=False, x=sol.t, y=sol.y[0], name='x', legendgroup='sol', mode='lines+markers'), row=1, col=1)
    fig.add_trace(go.Scatter(visible=False, x=sol.t, y=err, name='|xexa-x|', legendgroup='err', mode='markers'), row=2, col=1)
    fig.add_trace(go.Bar(visible=False, x=sol.t[:-1]+0.5*dt, y=dt, width=dt, name='dt', showlegend=False, marker_line_width=0.05), row=3, col=1)
    
for i in range(3): fig.data[i].visible = True

steps = []
for i, eps_i in enumerate(eps):
    args = [{"visible": [(el>=3*i) and (el<=3*i+2) for el in range(len(fig.data))]}]
    step = dict(method="update", label = f"{eps_i}", args=args)
    steps.append(step)
sliders = [dict(currentvalue={"prefix": "eps : "}, steps=steps)]

legend = dict(x=0.0, bgcolor='rgba(0,0,0,0)', tracegroupgap=280)
fig.update_layout(legend=legend, height=1000, sliders=sliders)
fig.update_xaxes(title_text='t', row=1)
fig.update_xaxes(title_text='t', row=2)
fig.update_yaxes(type="log", exponentformat='e', row=2)
fig.update_xaxes(title_text='t', row=3)
fig.update_yaxes(title_text='dt', row=3)
fig.show()

### Method of order 8

In order to compute the global error, the quasi-exact solution is obtained by using an explicit Runge-Kutta method of order 8 with stepsize control and fine tolerances due to Dormand and Prince.

In [ ]:
tini = 0. 
tend = 10.

yini = (0.5, 0)

tol = 1e-6

fig = make_subplots(rows=3, cols=1, vertical_spacing=0.1,
                    subplot_titles=(f"Solution x", f"Global error (tol={tol})", f"Time steps (tol={tol})"))

eps = np.arange(1,21)
for eps_i in eps:
    vdpm = vanderpol_model(eps=eps_i)
    fcn = vdpm.fcn
    jac = vdpm.jac
    sol = solve_ivp(fcn, (tini, tend), yini, method="DOP853", rtol=tol, atol=tol)
    sol_exa = solve_ivp(fcn, (tini, tend), yini, method="DOP853", rtol=1e-09, atol=1e-12, t_eval=sol.t)
    err = np.abs(sol_exa.y[0]-sol.y[0])
    dt = sol.t[1:]-sol.t[:-1]
    
    fig.add_trace(go.Scatter(visible=False, x=sol.t, y=sol.y[0], name='x', legendgroup='sol', mode='lines+markers'), row=1, col=1)
    fig.add_trace(go.Scatter(visible=False, x=sol.t, y=err, name='|xexa-x|', legendgroup='err', mode='markers'), row=2, col=1)
    fig.add_trace(go.Bar(visible=False, x=sol.t[:-1]+0.5*dt, y=dt, width=dt, name='dt', showlegend=False, marker_line_width=0.05), row=3, col=1)
    
for i in range(3): fig.data[i].visible = True

steps = []
for i, eps_i in enumerate(eps):
    args = [{"visible": [(el>=3*i) and (el<=3*i+2) for el in range(len(fig.data))]}]
    step = dict(method="update", label = f"{eps_i}", args=args)
    steps.append(step)
sliders = [dict(currentvalue={"prefix": "eps : "}, steps=steps)]

legend = dict(x=0.0, bgcolor='rgba(0,0,0,0)', tracegroupgap=280)
fig.update_layout(legend=legend, height=1000, sliders=sliders)
fig.update_xaxes(title_text='t', row=1)
fig.update_xaxes(title_text='t', row=2)
fig.update_yaxes(type="log", exponentformat='e', row=2)
fig.update_xaxes(title_text='t', row=3)
fig.update_yaxes(title_text='dt', row=3)
fig.show()